# einops preactice

可読性と信頼性が高いコードのためのテンソル演算ライブラリ。
Numpy, PyTorch, TensorFlow, JAXをサポート。
einopsの公式ドキュメント(https://einops.rocks/)
### install
```
pip install torch
pip install einops
```


In [2]:
import torch
from einops import rearrange, reduce, repeat, pack, unpack, einsum

### pack
任意の次元でテンソルを連結する。
#### 使い方
- 三次元のテンソル`tensor1`と`tensor2`がある場合、以下のように実行すると、三次元目で連結される。
```
c, ps = pack((tensor1, tensor2), 'a b *')
```
- `*`の次元で連結される
- 連結次元の指定に用いる文字はなんでもOK
  - 用途に応じてわかりやすい命名を
- 連結次元はサイズが異なっていても問題ない
- psには元のテンソルの情報が保存される

In [12]:
# example tensor
batch_size = 128
patch_num = 14 * 14
dim = 786
a = torch.randn((batch_size, patch_num, dim))
b = torch.randn((batch_size, patch_num, dim))

In [4]:
c, ps = pack((a, b), 'b * d')
c.shape

torch.Size([128, 392, 786])

In [5]:
c, ps = pack((a, b), '* p d')
c.shape

torch.Size([256, 196, 786])

In [6]:
c, ps = pack((a, b), 'b p *')
c.shape

torch.Size([128, 196, 1572])

In [7]:
c, ps = pack((a, b), 'a b *')
c.shape

torch.Size([128, 196, 1572])

In [13]:
d , ps = pack((a ,b, b), 'a b *')
d.shape

torch.Size([128, 196, 2358])

In [15]:
d , ps = pack((a ,b ,c), 'a b *')
d.shape

torch.Size([128, 196, 3144])

In [12]:
ps

[torch.Size([196]), torch.Size([196])]

### unpack
- packで連結したテンソルを分割する
#### 使い方
- packで`t1`と`t2`の二つのテンソルを連結したテンソル`packed_tensor`がある場合
```
t1, t2 = unpack(packed_tensor, ps, 'a b *')
```
とすることで元のテンソルに分割できる

In [17]:
# example tensor
batch_size = 128
patch_num = 14 * 14
dim = 786
a = torch.randn((batch_size, patch_num, dim))
b = torch.randn((batch_size, patch_num, dim))
c, ps = pack((a, b), 'b * d')

In [20]:
t1, t2 = unpack(c, ps, 'b * d')
print(c.shape)
print(t1.shape)
print(t2.shape)

torch.Size([128, 392, 786])
torch.Size([128, 196, 786])
torch.Size([128, 196, 786])


### reduce
- 指定した方法でテンソルの次元を削減する
#### 使い方

In [21]:
# example tensor
batch_size = 128
patch_num = 14 * 14
dim = 786
a = torch.randn((batch_size, patch_num, dim))
b = torch.ones((batch_size, patch_num, dim))

In [22]:
c = reduce(a, 'b p d -> b d', 'mean')
c.shape

torch.Size([128, 786])

In [26]:
d = reduce(b, 'b p d -> b d', 'mean')
d.shape

torch.Size([128, 786])